# Brown bear habitat suitability modelling

This notebook builds a reproducible species distribution modelling workflow for
female brown bears with cubs in the Pyrenees. It uses GPS telemetry as the
training signal, field observations as independent validation, and dynamically
discovers the environmental raster stack generated by `pirineus-raster`.

The workflow follows the CARTOBIO 2023 report structure but replaces MaxEnt with
tree-based models. XGBoost, Optuna, and SHAP are treated as optional modelling
extras: if they are installed, the notebook runs the primary XGBoost workflow;
if they are missing, the Random Forest baseline still runs and the missing steps
are logged clearly.


## 0. Parameters

These values are intentionally centralized so the notebook can be executed from
Jupyter or from SLURM via papermill. The raster directory is discovered at run
time, so new features added by the running raster job are included automatically
on the next execution.


In [ ]:
# Parameters (papermill)
OUTPUT_DIR = "outputs/ursus_arctos_habitat_modelling"
N_OPTUNA_TRIALS = 100
N_BACKGROUND = 10_000
RANDOM_SEED = 42
N_THIN_ITERATIONS = 20
MIN_THIN_DISTANCE_M = 100
BATCH_SIZE = 100_000

RUN_TUNING = False
RUN_MAP_PREDICTION = False
RUN_UNCERTAINTY_MAPS = False
RUN_SHAP = False
RUN_OBS_PARALLEL_MODELS = False

RASTER_DIR = "data_processed/datasets/ursus_arctos_pyrenees_100m/rasters"
GPS_PATH = "data_processed/notebooks/ursus_arctos_project/GPS_female_bears_with_cubs.csv"
OBS_PATH = "data_processed/notebooks/ursus_arctos_project/observations_female_bears_with_cubs.csv"
INDIVIDUALS_PATH = "data_processed/notebooks/ursus_arctos_project/bear_individuals.csv"
GRID_PATH = "data_interim/grids/grid_ursus_arctos_pyrenees_100m.tif"
AOI_CONFIG_PATH = "configs/aoi/ursus_arctos_pyrenees.yaml"
RUN_CONFIG_PATH = "configs/runs/ursus_arctos_pyrenees_100m.yaml"
MAXENT_REFERENCE_DIR = "use_cases_info"


## 1. Environment setup

This section finds the repository root, creates output folders, configures
logging, and checks optional packages. The logging setup writes to both the
notebook output and a file under `outputs/.../logs/`.


In [ ]:
import importlib
import json
import logging
import math
import os
import pickle
import random
import sys
import time
import warnings
from dataclasses import dataclass
from pathlib import Path

def find_project_root(start: Path) -> Path:
    """Return the repository root by walking upward from a notebook path."""
    start = start.resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() and (candidate / "configs").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing pyproject.toml and configs/")

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)

OUTPUT_ROOT = (PROJECT_ROOT / OUTPUT_DIR).resolve()
MAP_DIR = OUTPUT_ROOT / "maps"
PLOT_DIR = OUTPUT_ROOT / "plots"
TABLE_DIR = OUTPUT_ROOT / "tables"
MODEL_DIR = OUTPUT_ROOT / "models"
LOG_DIR = OUTPUT_ROOT / "logs"
INTERMEDIATE_DIR = OUTPUT_ROOT / "intermediate"
for directory in [OUTPUT_ROOT, MAP_DIR, PLOT_DIR, TABLE_DIR, MODEL_DIR, LOG_DIR, INTERMEDIATE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("MPLCONFIGDIR", "/tmp/pirineus_raster_matplotlib")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

logging.captureWarnings(True)
logger = logging.getLogger("ursus_arctos_sdm")
logger.setLevel(logging.INFO)
logger.handlers.clear()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setFormatter(formatter)
file_handler = logging.FileHandler(LOG_DIR / "sdm_bears.log")
file_handler.setFormatter(formatter)
logger.addHandler(stream_handler)
logger.addHandler(file_handler)

warnings.filterwarnings("ignore", category=FutureWarning)
random.seed(RANDOM_SEED)

logger.info("Project root: %s", PROJECT_ROOT)
logger.info("Output root: %s", OUTPUT_ROOT)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import ndimage
from scipy.spatial import cKDTree
from scipy.stats import mannwhitneyu, pearsonr, spearmanr

import rasterio
from rasterio.enums import Resampling
from rasterio.transform import xy
from rasterio.vrt import WarpedVRT
from rasterio.warp import reproject
from rasterio.windows import Window
from pyproj import Transformer

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    cohen_kappa_score,
    log_loss,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import ParameterSampler, train_test_split

try:
    from IPython.display import display
except Exception:  # pragma: no cover - notebook convenience fallback
    display = print

try:
    import yaml
except Exception as exc:
    raise ImportError("PyYAML is required to read the AOI config.") from exc

def optional_import(module_name: str):
    """Import an optional module and return `(module, available_bool)`."""
    try:
        module = importlib.import_module(module_name)
        logger.info("Optional package available: %s %s", module_name, getattr(module, "__version__", ""))
        return module, True
    except Exception as exc:
        logger.warning("Optional package missing: %s (%s)", module_name, exc)
        return None, False

xgboost, HAS_XGBOOST = optional_import("xgboost")
optuna, HAS_OPTUNA = optional_import("optuna")
shap, HAS_SHAP = optional_import("shap")
joblib, HAS_JOBLIB = optional_import("joblib")

if RUN_SHAP and not (HAS_XGBOOST and HAS_SHAP):
    logger.warning("RUN_SHAP=True but xgboost/shap is unavailable; SHAP cells will be skipped.")
if RUN_TUNING and not (HAS_XGBOOST and HAS_OPTUNA):
    logger.warning("RUN_TUNING=True but xgboost/optuna is unavailable; XGBoost tuning will be skipped.")


## 2. Repository scan

The modelling notebook should not rely on a hand-written raster list. This scan
documents all GeoTIFFs, YAML configs, reference model files, and SLURM logs found
under the repository. A CSV copy is saved for reproducibility.


In [ ]:
def scan_repository(root: Path) -> pd.DataFrame:
    """Scan the repository for files relevant to the habitat modelling workflow."""
    suffix_groups = {
        "raster": {".tif", ".tiff"},
        "config": {".yaml", ".yml"},
        "log": {".out", ".err", ".log"},
        "notebook": {".ipynb"},
    }
    records = []
    skip_parts = {".git", ".ipynb_checkpoints", "__pycache__"}
    for path in root.rglob("*"):
        if not path.is_file() or any(part in skip_parts for part in path.parts):
            continue
        suffix = path.suffix.lower()
        category = None
        for name, suffixes in suffix_groups.items():
            if suffix in suffixes:
                category = name
                break
        if category is None and "model" not in path.name.lower() and "maxent" not in path.name.lower():
            continue
        records.append(
            {
                "category": category or "model_reference",
                "path": str(path.relative_to(root)),
                "size_mb": path.stat().st_size / 1_000_000,
                "modified_utc": pd.to_datetime(path.stat().st_mtime, unit="s", utc=True),
            }
        )
    return pd.DataFrame(records).sort_values(["category", "path"]).reset_index(drop=True)

repo_scan = scan_repository(PROJECT_ROOT)
repo_scan.to_csv(TABLE_DIR / "repository_scan.csv", index=False)
logger.info("Repository scan saved with %d relevant files.", len(repo_scan))
display(repo_scan)


## 3. Paths, AOI, raster inventory, and reference rasters

The raster inventory reads every `.tif` currently present in the environmental
dataset directory. Companion JSON metadata are joined when available. If the
background raster job adds new features later, re-running this section will pick
them up.


In [ ]:
def resolve_project_path(path_like: str | Path) -> Path:
    """Resolve a relative path against the project root."""
    path = Path(path_like)
    return path if path.is_absolute() else PROJECT_ROOT / path

RASTER_DIR = resolve_project_path(RASTER_DIR)
GPS_PATH = resolve_project_path(GPS_PATH)
OBS_PATH = resolve_project_path(OBS_PATH)
INDIVIDUALS_PATH = resolve_project_path(INDIVIDUALS_PATH)
GRID_PATH = resolve_project_path(GRID_PATH)
AOI_CONFIG_PATH = resolve_project_path(AOI_CONFIG_PATH)
RUN_CONFIG_PATH = resolve_project_path(RUN_CONFIG_PATH)
MAXENT_REFERENCE_DIR = resolve_project_path(MAXENT_REFERENCE_DIR) if MAXENT_REFERENCE_DIR else None

required_paths = [RASTER_DIR, GPS_PATH, OBS_PATH, INDIVIDUALS_PATH, GRID_PATH, AOI_CONFIG_PATH, RUN_CONFIG_PATH]
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Missing required inputs: {missing_paths}")

with AOI_CONFIG_PATH.open("r", encoding="utf-8") as f:
    AOI_CONFIG = yaml.safe_load(f)
TARGET_CRS = AOI_CONFIG["crs"]
AOI_BOUNDS_4326 = AOI_CONFIG.get("bounds_epsg4326", {})

logger.info("Target CRS: %s", TARGET_CRS)
logger.info("AOI bounds EPSG:4326: %s", AOI_BOUNDS_4326)


In [ ]:
def feature_name_from_path(path: Path) -> str:
    """Create a compact, stable feature name from a raster path."""
    name = path.stem
    for suffix in ["_ursus_arctos_pyrenees_100m", "_pyrenees_100m", "_100m"]:
        if name.endswith(suffix):
            name = name[: -len(suffix)]
    return name

def read_json_metadata(path: Path) -> dict:
    """Read companion raster metadata, returning an empty dict if unavailable."""
    json_path = path.with_suffix(".json")
    if not json_path.exists():
        return {}
    try:
        with json_path.open("r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as exc:
        logger.warning("Could not read metadata %s: %s", json_path, exc)
        return {}

def load_raster_inventory(raster_dir: Path) -> pd.DataFrame:
    """Discover environmental rasters and attach lightweight metadata."""
    records = []
    for path in sorted([*raster_dir.glob("*.tif"), *raster_dir.glob("*.tiff")]):
        metadata = read_json_metadata(path)
        with rasterio.open(path) as src:
            records.append(
                {
                    "feature": feature_name_from_path(path),
                    "path": str(path),
                    "json_path": str(path.with_suffix(".json")) if path.with_suffix(".json").exists() else None,
                    "provider": metadata.get("provider"),
                    "product": metadata.get("product"),
                    "variable": metadata.get("variable"),
                    "description": metadata.get("variable_description") or metadata.get("description"),
                    "unit": metadata.get("unit"),
                    "value_semantics": metadata.get("value_semantics"),
                    "crs": str(src.crs),
                    "width": src.width,
                    "height": src.height,
                    "res_x": src.res[0],
                    "res_y": src.res[1],
                    "nodata": src.nodata,
                    "dtype": src.dtypes[0],
                    "size_mb": path.stat().st_size / 1_000_000,
                    "modified_utc": pd.to_datetime(path.stat().st_mtime, unit="s", utc=True),
                }
            )
    inventory = pd.DataFrame(records)
    if inventory.empty:
        raise FileNotFoundError(f"No raster features found in {raster_dir}")
    if inventory["feature"].duplicated().any():
        duplicated = inventory.loc[inventory["feature"].duplicated(), "feature"].tolist()
        raise ValueError(f"Duplicated feature names after normalization: {duplicated}")
    return inventory

raster_inventory = load_raster_inventory(RASTER_DIR)
raster_inventory.to_csv(TABLE_DIR / "environmental_raster_inventory.csv", index=False)
logger.info("Discovered %d environmental raster features.", len(raster_inventory))
display(raster_inventory[["feature", "provider", "variable", "unit", "width", "height", "crs", "modified_utc"]])


In [ ]:
def check_raster_alignment(inventory: pd.DataFrame, grid_path: Path) -> pd.DataFrame:
    """Compare every feature raster to the project grid geometry."""
    records = []
    with rasterio.open(grid_path) as grid:
        for row in inventory.itertuples(index=False):
            with rasterio.open(row.path) as src:
                same_shape = (src.width == grid.width) and (src.height == grid.height)
                same_transform = src.transform.almost_equals(grid.transform)
                same_bounds = np.allclose(tuple(src.bounds), tuple(grid.bounds), atol=1e-6)
                records.append(
                    {
                        "feature": row.feature,
                        "same_shape": same_shape,
                        "same_transform": same_transform,
                        "same_bounds": same_bounds,
                        "grid_crs": str(grid.crs),
                        "raster_crs": str(src.crs),
                    }
                )
    alignment = pd.DataFrame(records)
    alignment.to_csv(TABLE_DIR / "raster_alignment_check.csv", index=False)
    return alignment

alignment = check_raster_alignment(raster_inventory, GRID_PATH)
if not alignment[["same_shape", "same_transform", "same_bounds"]].all(axis=None):
    logger.warning("Some environmental rasters differ from the project grid. Review raster_alignment_check.csv.")
display(alignment)

REFERENCE_MAXENT = {
    "lt6": {
        "label": "females_cubs_0to6_months",
        "model": MAXENT_REFERENCE_DIR / "ursus_arctos_females_cubs_0to6_months_v2023_model_potencial.tif",
        "zones": MAXENT_REFERENCE_DIR / "ursus_arctos_females_cubs_0to6_months_v2023_zones_potencial.tif",
    },
    "6to12": {
        "label": "females_cubs_6to12_months",
        "model": MAXENT_REFERENCE_DIR / "ursus_arctos_females_cubs_6to12_months_v2023_model_potencial.tif",
        "zones": MAXENT_REFERENCE_DIR / "ursus_arctos_females_cubs_6to12_months_v2023_zones_potencial.tif",
    },
}
for cub_class, refs in REFERENCE_MAXENT.items():
    for kind, path in refs.items():
        if kind != "label" and not path.exists():
            logger.warning("Missing MaxEnt reference for %s %s: %s", cub_class, kind, path)


## 4. Biological data loading and cleaning

GPS telemetry is the training source because collar fixes represent where each
tracked bear moved, while field observations are kept for independent validation
because they can reflect sampling effort. Cleaning is reported step by step.


In [ ]:
CUB_CLASS_MAP = {
    "<6month": "lt6",
    "1": "6to12",
    "1.0": "6to12",
}

def recode_cub_class(values: pd.Series) -> pd.Series:
    """Recode project cub-age labels into the two model classes."""
    normalized = values.astype("string").str.strip()
    return normalized.map(CUB_CLASS_MAP).astype("string")

def bounds_filter(df: pd.DataFrame, lon_col: str = "x_long", lat_col: str = "y_lat") -> pd.Series:
    """Return True for points inside the AOI longitude/latitude bounds."""
    xmin = AOI_BOUNDS_4326.get("xmin", -2.0)
    xmax = AOI_BOUNDS_4326.get("xmax", 3.5)
    ymin = AOI_BOUNDS_4326.get("ymin", 42.0)
    ymax = AOI_BOUNDS_4326.get("ymax", 43.6)
    lon = pd.to_numeric(df[lon_col], errors="coerce")
    lat = pd.to_numeric(df[lat_col], errors="coerce")
    return lon.between(xmin, xmax) & lat.between(ymin, ymax)

def reproject_lonlat(df: pd.DataFrame, lon_col: str = "x_long", lat_col: str = "y_lat") -> pd.DataFrame:
    """Add projected coordinates in the modelling CRS from WGS84 lon/lat."""
    transformer = Transformer.from_crs("EPSG:4326", TARGET_CRS, always_xy=True)
    lon = pd.to_numeric(df[lon_col], errors="coerce").to_numpy()
    lat = pd.to_numeric(df[lat_col], errors="coerce").to_numpy()
    x, y = transformer.transform(lon, lat)
    out = df.copy()
    out["x_3035"] = x
    out["y_3035"] = y
    return out

def clean_gps_data(gps_path: Path, individuals_path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load and clean GPS telemetry for model training.

    The cleaning removes duplicates, points outside the study-area bounds, rows
    without the two target cub-age classes, and records after a bear's known
    mortality or disappearance year.
    """
    gps = pd.read_csv(gps_path)
    individuals = pd.read_csv(individuals_path)
    report = []

    def record(step: str, before: int, after: int) -> None:
        report.append({"step": step, "before": before, "after": after, "removed": before - after})

    before = len(gps)
    gps = gps.drop_duplicates(subset=["id_obs"]).copy()
    record("drop duplicated id_obs", before, len(gps))

    gps["datetime_utc"] = pd.to_datetime(
        gps["date_gmt"].astype("string").fillna("") + " " + gps["time_gmt"].astype("string").fillna("00:00:00"),
        errors="coerce",
        utc=True,
    )
    gps["event_year"] = pd.to_numeric(gps["year"], errors="coerce")
    parsed_year = gps["datetime_utc"].dt.year
    gps.loc[parsed_year.notna(), "event_year"] = parsed_year[parsed_year.notna()]
    gps["cub_class"] = recode_cub_class(gps["with_cubs_estimated"])

    before = len(gps)
    gps = gps[gps["cub_class"].notna()].copy()
    record("keep target cub classes", before, len(gps))

    before = len(gps)
    gps = gps[bounds_filter(gps)].copy()
    record("keep points inside AOI lon/lat bounds", before, len(gps))

    registry = individuals[["code", "name", "mortality_year", "suposed_desaparition_year", "disappeared"]].copy()
    registry["name_key"] = registry["name"].astype("string").str.lower().str.strip()
    gps["name_key"] = gps["bear_name"].astype("string").str.lower().str.strip()
    missing = sorted(set(gps["name_key"].dropna()) - set(registry["name_key"].dropna()))
    if missing:
        raise ValueError(f"GPS bear names not found in registry: {missing}")

    gps = gps.merge(
        registry,
        on="name_key",
        how="left",
        suffixes=("", "_registry"),
    )
    gps["end_year"] = gps[["mortality_year", "suposed_desaparition_year"]].min(axis=1, skipna=True)
    before = len(gps)
    gps = gps[~(gps["end_year"].notna() & gps["event_year"].notna() & (gps["event_year"] > gps["end_year"]))].copy()
    record("remove records after mortality/disappearance year", before, len(gps))

    gps = reproject_lonlat(gps)
    report_df = pd.DataFrame(report)
    return gps.reset_index(drop=True), report_df

def clean_obs_data(obs_path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load field observations for independent validation only."""
    obs = pd.read_csv(obs_path)
    report = []
    before = len(obs)
    obs = obs.drop_duplicates(subset=["id_obs"]).copy()
    report.append({"step": "drop duplicated id_obs", "before": before, "after": len(obs), "removed": before - len(obs)})

    obs["date_parsed"] = pd.to_datetime(obs["date"], errors="coerce", dayfirst=True, utc=True)
    obs["event_year"] = pd.to_numeric(obs["year"], errors="coerce")
    parsed_year = obs["date_parsed"].dt.year
    obs.loc[parsed_year.notna(), "event_year"] = parsed_year[parsed_year.notna()]
    obs["cub_class"] = recode_cub_class(obs["with_cubs_estimated"])

    before = len(obs)
    obs = obs[obs["cub_class"].notna()].copy()
    report.append({"step": "keep target cub classes", "before": before, "after": len(obs), "removed": before - len(obs)})

    before = len(obs)
    obs = obs[bounds_filter(obs)].copy()
    report.append({"step": "keep points inside AOI lon/lat bounds", "before": before, "after": len(obs), "removed": before - len(obs)})

    obs = reproject_lonlat(obs)
    return obs.reset_index(drop=True), pd.DataFrame(report)

gps_clean, gps_cleaning_report = clean_gps_data(GPS_PATH, INDIVIDUALS_PATH)
obs_clean, obs_cleaning_report = clean_obs_data(OBS_PATH)
gps_cleaning_report.to_csv(TABLE_DIR / "gps_cleaning_report.csv", index=False)
obs_cleaning_report.to_csv(TABLE_DIR / "obs_cleaning_report.csv", index=False)

logger.info("GPS cleaned rows: %d", len(gps_clean))
logger.info("OBS cleaned rows: %d", len(obs_clean))
display(gps_cleaning_report)
display(obs_cleaning_report)
display(pd.crosstab(gps_clean["bear_name"], gps_clean["cub_class"]))
display(pd.crosstab(obs_clean["confirmed_individual"], obs_clean["cub_class"]).sort_index())


## 5. Spatial thinning

GPS fixes can cluster strongly when a bear remains in the same area. The thinning
step keeps the largest random greedy subset per individual where no two retained
points are closer than the 100 m modelling resolution.


In [ ]:
def spatial_thin_one(
    df: pd.DataFrame,
    min_dist_m: float = 100,
    n_iterations: int = 20,
    random_state: int = 42,
) -> pd.DataFrame:
    """Thin one set of projected points using random greedy independent sets.

    Points closer than `min_dist_m` compete; repeated random orderings are tried,
    and the retained set with the largest number of points is returned.
    """
    if len(df) <= 1:
        return df.copy()
    coords = df[["x_3035", "y_3035"]].to_numpy(dtype=float)
    valid = np.isfinite(coords).all(axis=1)
    if not valid.all():
        df = df.loc[valid].copy()
        coords = coords[valid]
    if len(df) <= 1:
        return df.copy()

    tree = cKDTree(coords)
    pairs = tree.query_pairs(r=max(float(min_dist_m) - 1e-9, 0.0))
    conflicts = [set() for _ in range(len(df))]
    for i, j in pairs:
        conflicts[i].add(j)
        conflicts[j].add(i)

    rng = np.random.default_rng(random_state)
    best_keep = None
    best_count = -1
    for _ in range(n_iterations):
        order = rng.permutation(len(df))
        blocked = np.zeros(len(df), dtype=bool)
        keep = np.zeros(len(df), dtype=bool)
        for idx in order:
            if blocked[idx]:
                continue
            keep[idx] = True
            if conflicts[idx]:
                blocked[list(conflicts[idx])] = True
        count = int(keep.sum())
        if count > best_count:
            best_count = count
            best_keep = keep
    return df.iloc[np.where(best_keep)[0]].copy()

def spatial_thin_by_group(
    df: pd.DataFrame,
    group_col: str,
    min_dist_m: float = 100,
    n_iterations: int = 20,
    random_state: int = 42,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Apply spatial thinning separately to each individual bear."""
    thinned_parts = []
    records = []
    for offset, (group, part) in enumerate(df.groupby(group_col, dropna=False)):
        thinned = spatial_thin_one(
            part,
            min_dist_m=min_dist_m,
            n_iterations=n_iterations,
            random_state=random_state + offset,
        )
        thinned_parts.append(thinned)
        records.append({"group": group, "before": len(part), "after": len(thinned), "removed": len(part) - len(thinned)})
    return pd.concat(thinned_parts, ignore_index=True), pd.DataFrame(records)

presence_sets = {}
thinning_reports = []
for cub_class in ["lt6", "6to12"]:
    subset = gps_clean[gps_clean["cub_class"] == cub_class].copy()
    thinned, report = spatial_thin_by_group(
        subset,
        group_col="bear_name",
        min_dist_m=MIN_THIN_DISTANCE_M,
        n_iterations=N_THIN_ITERATIONS,
        random_state=RANDOM_SEED,
    )
    presence_sets[cub_class] = thinned.reset_index(drop=True)
    report.insert(0, "cub_class", cub_class)
    thinning_reports.append(report)
    logger.info("%s: thinned GPS presences from %d to %d", cub_class, len(subset), len(thinned))

thinning_report = pd.concat(thinning_reports, ignore_index=True)
thinning_report.to_csv(TABLE_DIR / "gps_spatial_thinning_report.csv", index=False)
display(thinning_report)


## 6. Background points and raster feature extraction

Background points are sampled from valid raster pixels within the study area.
They are not confirmed absences; they represent available environmental
conditions against which GPS presences are contrasted.


In [ ]:
def choose_mask_raster(inventory: pd.DataFrame) -> Path:
    """Choose a stable raster to define the valid study-area mask."""
    preferred_terms = ["derived_topography_dem_elevation", "copernicus_dem_glo30_elevation", "elevation"]
    for term in preferred_terms:
        matches = inventory[inventory["feature"].str.contains(term, case=False, regex=False)]
        if not matches.empty:
            return Path(matches.iloc[0]["path"])
    return Path(inventory.iloc[0]["path"])

MASK_RASTER_PATH = choose_mask_raster(raster_inventory)
logger.info("Background sampling mask raster: %s", MASK_RASTER_PATH)

def generate_background(mask_raster_path: Path, n_points: int = 10_000, random_state: int = 42) -> pd.DataFrame:
    """Sample random background points from valid cells of a raster mask.

    Background points are sampled at pixel centers in the target projected CRS.
    """
    rng = np.random.default_rng(random_state)
    with rasterio.open(mask_raster_path) as src:
        arr = src.read(1, masked=True)
        valid = ~np.ma.getmaskarray(arr)
        values = np.asarray(arr.filled(np.nan), dtype=float)
        if src.nodata is not None:
            valid &= ~np.isclose(values, src.nodata)
        valid &= np.isfinite(values)
        rows, cols = np.where(valid)
        if rows.size == 0:
            raise ValueError(f"No valid pixels found in mask raster: {mask_raster_path}")
        replace = n_points > rows.size
        selected = rng.choice(rows.size, size=n_points, replace=replace)
        sampled_rows = rows[selected]
        sampled_cols = cols[selected]
        xs, ys = xy(src.transform, sampled_rows, sampled_cols, offset="center")
    return pd.DataFrame(
        {
            "point_id": [f"background_{i}" for i in range(n_points)],
            "x_3035": np.asarray(xs, dtype=float),
            "y_3035": np.asarray(ys, dtype=float),
            "source": "background",
            "label": 0,
            "bear_name": "background",
        }
    )

background_points = generate_background(MASK_RASTER_PATH, n_points=N_BACKGROUND, random_state=RANDOM_SEED)
background_points.to_csv(INTERMEDIATE_DIR / "background_points.csv", index=False)
logger.info("Generated %d background points.", len(background_points))
display(background_points.head())


In [ ]:
def infer_log_transform_columns(feature_names: list[str]) -> list[str]:
    """Detect raw distance features that should be log10(x + 1) transformed."""
    log_cols = []
    for name in feature_names:
        low = name.lower()
        is_distance = "distance" in low or "dist_" in low or low.endswith("_dist")
        already_log = "logdistance" in low or "log_distance" in low or "log10" in low
        if is_distance and not already_log:
            log_cols.append(name)
    return log_cols

def extract_raster_values(points: pd.DataFrame, inventory: pd.DataFrame) -> pd.DataFrame:
    """Extract environmental raster values at projected point coordinates.

    Raster sampling avoids loading the full stack into memory. NoData and
    non-finite values are converted to NaN for later filtering and imputation.
    """
    coords = list(zip(points["x_3035"].astype(float), points["y_3035"].astype(float)))
    feature_values = {}
    for row in inventory.itertuples(index=False):
        path = Path(row.path)
        with rasterio.open(path) as src:
            values = np.array([sample[0] for sample in src.sample(coords)], dtype="float64")
            if src.nodata is not None:
                values[np.isclose(values, src.nodata)] = np.nan
            values[~np.isfinite(values)] = np.nan
        feature_values[row.feature] = values
    return pd.DataFrame(feature_values, index=points.index)

def apply_feature_transforms(features: pd.DataFrame, log_columns: list[str]) -> pd.DataFrame:
    """Apply deterministic transforms to raw feature values."""
    transformed = features.copy()
    for col in log_columns:
        if col not in transformed.columns:
            continue
        values = pd.to_numeric(transformed[col], errors="coerce").astype(float)
        values = values.where(values >= 0)
        transformed[col] = np.log10(values + 1.0)
    return transformed

def prepare_training_table(
    presences: pd.DataFrame,
    background: pd.DataFrame,
    inventory: pd.DataFrame,
    cub_class: str,
    nan_row_threshold: float = 0.20,
) -> dict:
    """Create a model-ready table for one cub-age class."""
    pres = presences.copy()
    pres["label"] = 1
    pres["source"] = "gps"
    pres["point_id"] = pres["id_obs"].astype("string").fillna(pd.Series(range(len(pres)), index=pres.index).astype(str))

    bg = background.copy()
    points = pd.concat(
        [
            pres[["point_id", "x_3035", "y_3035", "source", "label", "bear_name"]],
            bg[["point_id", "x_3035", "y_3035", "source", "label", "bear_name"]],
        ],
        ignore_index=True,
    )
    raw_features = extract_raster_values(points, inventory)
    feature_cols = list(raw_features.columns)
    log_columns = infer_log_transform_columns(feature_cols)
    transformed = apply_feature_transforms(raw_features, log_columns)

    feature_nan = transformed.isna().mean().sort_values(ascending=False)
    flagged_features = feature_nan[feature_nan > 0.30]
    if len(flagged_features):
        logger.warning("%s: features with >30%% NaN: %s", cub_class, flagged_features.to_dict())
    empty_features = feature_nan[feature_nan >= 1.0].index.tolist()
    if empty_features:
        logger.warning("%s: dropping fully empty feature rasters before modelling: %s", cub_class, empty_features)
        transformed = transformed.drop(columns=empty_features)

    feature_cols = list(transformed.columns)
    log_columns = [col for col in log_columns if col in feature_cols]
    nan_fraction = transformed.isna().mean(axis=1)
    keep_rows = nan_fraction <= nan_row_threshold

    points_kept = points.loc[keep_rows].reset_index(drop=True)
    features_kept = transformed.loc[keep_rows].reset_index(drop=True)
    medians = features_kept.median(axis=0, skipna=True)
    still_empty = medians[medians.isna()].index.tolist()
    if still_empty:
        logger.warning("%s: dropping features with no median after row filtering: %s", cub_class, still_empty)
        features_kept = features_kept.drop(columns=still_empty)
        medians = medians.drop(index=still_empty)
        feature_cols = [col for col in feature_cols if col not in still_empty]
        log_columns = [col for col in log_columns if col in feature_cols]
    features_filled = features_kept.fillna(medians)

    table = pd.concat([points_kept.reset_index(drop=True), features_filled.reset_index(drop=True)], axis=1)
    table["cub_class"] = cub_class
    logger.info(
        "%s: built training table with %d rows (%d presences, %d background).",
        cub_class,
        len(table),
        int((table["label"] == 1).sum()),
        int((table["label"] == 0).sum()),
    )
    return {
        "table": table,
        "feature_cols": feature_cols,
        "log_columns": log_columns,
        "medians": medians,
        "feature_nan": feature_nan,
    }

model_data = {}
for cub_class in ["lt6", "6to12"]:
    model_data[cub_class] = prepare_training_table(
        presence_sets[cub_class],
        background_points,
        raster_inventory,
        cub_class,
    )
    model_data[cub_class]["table"].to_csv(INTERMEDIATE_DIR / f"training_table_{cub_class}.csv", index=False)
    model_data[cub_class]["feature_nan"].rename("nan_fraction").to_csv(TABLE_DIR / f"feature_nan_{cub_class}.csv")
    logger.info("%s log-transformed columns: %s", cub_class, model_data[cub_class]["log_columns"])


## 7. Exploratory data analysis

The EDA cells summarize spatial coverage, feature distributions, feature
correlations, and per-individual extents. They are designed to be useful before
heavy model tuning starts.


In [ ]:
def add_scalebar(ax, length_km: float = 50, pad_fraction: float = 0.06) -> None:
    """Draw a simple projected-coordinate scalebar on a matplotlib axis."""
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    length_m = length_km * 1000
    x0 = xmin + (xmax - xmin) * pad_fraction
    y0 = ymin + (ymax - ymin) * pad_fraction
    ax.plot([x0, x0 + length_m], [y0, y0], color="black", linewidth=3)
    ax.text(x0 + length_m / 2, y0 + (ymax - ymin) * 0.015, f"{length_km:g} km", ha="center", va="bottom")

def read_raster_preview(path: Path, max_size: int = 900) -> tuple[np.ma.MaskedArray, tuple[float, float, float, float]]:
    """Read a downsampled raster preview and return array plus plotting extent."""
    with rasterio.open(path) as src:
        scale = max(src.width / max_size, src.height / max_size, 1)
        out_height = max(1, int(src.height / scale))
        out_width = max(1, int(src.width / scale))
        arr = src.read(1, out_shape=(out_height, out_width), masked=True, resampling=Resampling.bilinear)
        extent = (src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top)
    return arr, extent

def plot_spatial_points(cub_class: str, presences: pd.DataFrame, background: pd.DataFrame, output_path: Path) -> None:
    """Plot thinned GPS presences and sampled background points over elevation."""
    dem, extent = read_raster_preview(MASK_RASTER_PATH)
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.imshow(dem, extent=extent, origin="upper", cmap="Greys", alpha=0.45)
    sample_bg = background.sample(min(2000, len(background)), random_state=RANDOM_SEED)
    ax.scatter(sample_bg["x_3035"], sample_bg["y_3035"], s=2, c="#999999", alpha=0.25, label="background")
    for bear, part in presences.groupby("bear_name"):
        ax.scatter(part["x_3035"], part["y_3035"], s=10, alpha=0.8, label=str(bear))
    ax.set_title(f"GPS presences and background: {cub_class}")
    ax.set_xlabel(f"X ({TARGET_CRS})")
    ax.set_ylabel(f"Y ({TARGET_CRS})")
    ax.legend(markerscale=2, fontsize=8, loc="upper right")
    add_scalebar(ax)
    fig.tight_layout()
    fig.savefig(output_path, dpi=300)
    plt.close(fig)

def plot_feature_boxplots(cub_class: str, table: pd.DataFrame, feature_cols: list[str], output_path: Path, max_features: int = 24) -> None:
    """Plot presence vs background feature distributions for a manageable subset."""
    selected = feature_cols[:max_features]
    long = table[["label", *selected]].melt(id_vars="label", var_name="feature", value_name="value")
    long["class"] = np.where(long["label"] == 1, "presence", "background")
    ncols = 4
    nrows = math.ceil(len(selected) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, max(3, nrows * 3)), squeeze=False)
    for ax, feature in zip(axes.ravel(), selected):
        data = [long[(long["feature"] == feature) & (long["class"] == cls)]["value"].dropna() for cls in ["presence", "background"]]
        ax.boxplot(data, labels=["presence", "background"], showfliers=False)
        ax.set_title(feature, fontsize=8)
        ax.tick_params(axis="x", labelrotation=20)
    for ax in axes.ravel()[len(selected):]:
        ax.axis("off")
    fig.suptitle(f"Feature distributions: {cub_class}", y=1.0)
    fig.tight_layout()
    fig.savefig(output_path, dpi=300)
    plt.close(fig)

def plot_correlation_heatmap(cub_class: str, table: pd.DataFrame, feature_cols: list[str], output_path: Path) -> pd.DataFrame:
    """Plot Spearman feature correlations and return highly correlated pairs."""
    corr = table[feature_cols].corr(method="spearman")
    fig, ax = plt.subplots(figsize=(12, 10))
    image = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
    ax.set_title(f"Spearman feature correlation: {cub_class}")
    ax.set_xticks(range(len(feature_cols)))
    ax.set_yticks(range(len(feature_cols)))
    ax.set_xticklabels(feature_cols, rotation=90, fontsize=5)
    ax.set_yticklabels(feature_cols, fontsize=5)
    fig.colorbar(image, ax=ax, fraction=0.03, pad=0.02, label="Spearman rho")
    fig.tight_layout()
    fig.savefig(output_path, dpi=300)
    plt.close(fig)

    pairs = []
    for i, a in enumerate(feature_cols):
        for j in range(i + 1, len(feature_cols)):
            b = feature_cols[j]
            rho = corr.iloc[i, j]
            if pd.notna(rho) and abs(rho) > 0.85:
                pairs.append({"cub_class": cub_class, "feature_a": a, "feature_b": b, "spearman_rho": rho})
    return pd.DataFrame(pairs)

def individual_summary(presences: pd.DataFrame) -> pd.DataFrame:
    """Summarize thinned point counts, date ranges, and approximate spatial extent."""
    records = []
    for bear, part in presences.groupby("bear_name"):
        x_range = part["x_3035"].max() - part["x_3035"].min()
        y_range = part["y_3035"].max() - part["y_3035"].min()
        records.append(
            {
                "bear_name": bear,
                "n_presences_after_thinning": len(part),
                "date_min": part["datetime_utc"].min(),
                "date_max": part["datetime_utc"].max(),
                "bbox_extent_km2": (x_range * y_range) / 1_000_000,
            }
        )
    return pd.DataFrame(records)

high_corr_tables = []
individual_tables = []
for cub_class, data in model_data.items():
    table = data["table"]
    feature_cols = data["feature_cols"]
    plot_spatial_points(cub_class, presence_sets[cub_class], background_points, PLOT_DIR / f"spatial_points_{cub_class}.png")
    plot_feature_boxplots(cub_class, table, feature_cols, PLOT_DIR / f"feature_distributions_{cub_class}.png")
    high_corr = plot_correlation_heatmap(cub_class, table, feature_cols, PLOT_DIR / f"feature_correlation_{cub_class}.png")
    high_corr_tables.append(high_corr)
    summary = individual_summary(presence_sets[cub_class])
    summary.insert(0, "cub_class", cub_class)
    individual_tables.append(summary)

high_corr_pairs = pd.concat(high_corr_tables, ignore_index=True) if high_corr_tables else pd.DataFrame()
individual_summary_table = pd.concat(individual_tables, ignore_index=True)
high_corr_pairs.to_csv(TABLE_DIR / "high_correlation_pairs.csv", index=False)
individual_summary_table.to_csv(TABLE_DIR / "individual_presence_summary.csv", index=False)
display(individual_summary_table)
display(high_corr_pairs)


## 8. Metrics and model utilities

Leave-One-Individual-Out cross-validation is the primary validation design. Each
fold withholds one GPS-collared female bear and uses a random 80/20 split of the
background points.


In [ ]:
def boyce_index(predicted_proba: np.ndarray, presence_mask: np.ndarray, n_bins: int = 10) -> float:
    """Compute the continuous Boyce Index for presence-background predictions."""
    predicted_proba = np.asarray(predicted_proba, dtype=float)
    presence_mask = np.asarray(presence_mask, dtype=bool)
    valid = np.isfinite(predicted_proba)
    predicted_proba = predicted_proba[valid]
    presence_mask = presence_mask[valid]
    if presence_mask.sum() < 2 or (~presence_mask).sum() < 2:
        return np.nan
    bins = np.linspace(np.nanmin(predicted_proba), np.nanmax(predicted_proba), n_bins + 1)
    if np.unique(bins).size < 3:
        return np.nan
    mids = []
    ratios = []
    for lo, hi in zip(bins[:-1], bins[1:]):
        in_bin = (predicted_proba >= lo) & (predicted_proba <= hi if hi == bins[-1] else predicted_proba < hi)
        expected = in_bin.mean()
        observed = in_bin[presence_mask].mean() if presence_mask.any() else np.nan
        if expected > 0 and np.isfinite(observed):
            mids.append((lo + hi) / 2)
            ratios.append(observed / expected)
    if len(ratios) < 3:
        return np.nan
    return float(spearmanr(mids, ratios).correlation)

def tss_at_optimal_threshold(y_true: np.ndarray, y_score: np.ndarray) -> tuple[float, float]:
    """Return True Skill Statistic and threshold maximizing Youden's J."""
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    youden = tpr - fpr
    best = int(np.nanargmax(youden))
    return float(youden[best]), float(thresholds[best])

def evaluate_predictions(y_true: np.ndarray, y_score: np.ndarray) -> dict:
    """Compute SDM evaluation metrics for probabilistic predictions."""
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    metrics = {
        "auc_roc": roc_auc_score(y_true, y_score) if np.unique(y_true).size == 2 else np.nan,
        "auc_pr": average_precision_score(y_true, y_score) if np.unique(y_true).size == 2 else np.nan,
        "boyce": boyce_index(y_score, y_true == 1),
        "log_loss": log_loss(y_true, np.clip(y_score, 1e-6, 1 - 1e-6), labels=[0, 1]) if np.unique(y_true).size == 2 else np.nan,
    }
    tss, threshold = tss_at_optimal_threshold(y_true, y_score) if np.unique(y_true).size == 2 else (np.nan, np.nan)
    metrics["tss"] = tss
    metrics["threshold_youden"] = threshold
    return metrics

def predict_positive_proba(model, X: pd.DataFrame | np.ndarray) -> np.ndarray:
    """Return class-1 probability from any supported classifier."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        scores = model.decision_function(X)
        return 1 / (1 + np.exp(-scores))
    raise TypeError(f"Model does not expose predict_proba or decision_function: {type(model)}")

def make_loio_folds(table: pd.DataFrame, random_state: int = 42) -> list[dict]:
    """Create Leave-One-Individual-Out folds with background split 80/20."""
    presence_groups = sorted(table.loc[table["label"] == 1, "bear_name"].dropna().unique())
    background_idx = table.index[table["label"] == 0].to_numpy()
    folds = []
    for fold_id, bear in enumerate(presence_groups):
        test_presence_idx = table.index[(table["label"] == 1) & (table["bear_name"] == bear)].to_numpy()
        train_presence_idx = table.index[(table["label"] == 1) & (table["bear_name"] != bear)].to_numpy()
        bg_train, bg_test = train_test_split(background_idx, test_size=0.20, random_state=random_state + fold_id)
        folds.append(
            {
                "fold": str(bear),
                "train_idx": np.concatenate([train_presence_idx, bg_train]),
                "test_idx": np.concatenate([test_presence_idx, bg_test]),
            }
        )
    return folds

def fit_model(algo: str, X_train: pd.DataFrame, y_train: pd.Series, params: dict):
    """Fit a supported model algorithm."""
    if algo == "rf":
        model = RandomForestClassifier(**params)
        return model.fit(X_train, y_train)
    if algo == "xgb":
        if not HAS_XGBOOST:
            raise ImportError("xgboost is not installed.")
        params = dict(params)
        params.setdefault("scale_pos_weight", max(1.0, float((y_train == 0).sum()) / max(1, int((y_train == 1).sum()))))
        model = xgboost.XGBClassifier(**params)
        return model.fit(X_train, y_train)
    raise ValueError(f"Unknown algorithm: {algo}")

def evaluate_loio(
    algo: str,
    params: dict,
    table: pd.DataFrame,
    feature_cols: list[str],
    cub_class: str,
    random_state: int = 42,
) -> tuple[pd.DataFrame, list]:
    """Train and evaluate one algorithm over LOIO folds."""
    folds = make_loio_folds(table, random_state=random_state)
    results = []
    fold_models = []
    for fold in folds:
        train = table.loc[fold["train_idx"]]
        test = table.loc[fold["test_idx"]]
        X_train = train[feature_cols]
        y_train = train["label"].astype(int)
        X_test = test[feature_cols]
        y_test = test["label"].astype(int)
        model = fit_model(algo, X_train, y_train, params)
        y_score = predict_positive_proba(model, X_test)
        metrics = evaluate_predictions(y_test, y_score)
        results.append({"cub_class": cub_class, "algorithm": algo, "fold": fold["fold"], **metrics})
        fold_models.append({"fold": fold["fold"], "model": model})
        logger.info("%s %s fold %s AUC=%.3f Boyce=%.3f", cub_class, algo, fold["fold"], metrics["auc_roc"], metrics["boyce"])
    result_df = pd.DataFrame(results)
    mean_row = result_df.select_dtypes(include=[np.number]).mean(numeric_only=True).to_dict()
    result_df = pd.concat(
        [result_df, pd.DataFrame([{**mean_row, "cub_class": cub_class, "algorithm": algo, "fold": "MEAN"}])],
        ignore_index=True,
    )
    return result_df, fold_models

DEFAULT_RF_PARAMS = {
    "n_estimators": 500,
    "max_depth": None,
    "max_features": "sqrt",
    "min_samples_leaf": 2,
    "class_weight": "balanced_subsample",
    "n_jobs": -1,
    "random_state": RANDOM_SEED,
}

DEFAULT_XGB_PARAMS = {
    "n_estimators": 500,
    "max_depth": 4,
    "learning_rate": 0.03,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 3,
    "reg_alpha": 0.01,
    "reg_lambda": 1.0,
    "tree_method": "hist",
    "eval_metric": "auc",
    "random_state": RANDOM_SEED,
    "n_jobs": -1,
}


## 9. Hyperparameter tuning

Tuning is optional because it is the slowest part of the workflow. Set
`RUN_TUNING=True` for the SLURM execution. XGBoost tuning uses Optuna when both
`xgboost` and `optuna` are installed. Random Forest uses a LOIO-aware randomized
search implemented directly over the same folds.


In [ ]:
def tune_rf_random_search(table: pd.DataFrame, feature_cols: list[str], cub_class: str, n_iter: int = 30) -> dict:
    """LOIO-aware randomized search for Random Forest hyperparameters."""
    param_dist = {
        "n_estimators": [200, 500, 1000],
        "max_depth": [None, 10, 20, 30],
        "max_features": ["sqrt", "log2", 0.3, 0.5],
        "min_samples_leaf": [1, 2, 5, 10],
        "class_weight": ["balanced", "balanced_subsample"],
    }
    sampler = list(ParameterSampler(param_dist, n_iter=n_iter, random_state=RANDOM_SEED))
    best_params = None
    best_auc = -np.inf
    for i, params in enumerate(sampler, start=1):
        params = {**params, "n_jobs": -1, "random_state": RANDOM_SEED}
        result_df, _ = evaluate_loio("rf", params, table, feature_cols, cub_class, random_state=RANDOM_SEED)
        auc = float(result_df.loc[result_df["fold"] == "MEAN", "auc_roc"].iloc[0])
        logger.info("%s RF tuning candidate %d/%d mean AUC=%.3f", cub_class, i, len(sampler), auc)
        if auc > best_auc:
            best_auc = auc
            best_params = params
    logger.info("%s RF best tuning AUC=%.3f params=%s", cub_class, best_auc, best_params)
    return best_params

def spatial_block_ids(table: pd.DataFrame, n_blocks_axis: int = 3) -> pd.Series:
    """Assign coarse spatial block ids for fast inner validation."""
    x_bins = pd.qcut(table["x_3035"], q=n_blocks_axis, labels=False, duplicates="drop")
    y_bins = pd.qcut(table["y_3035"], q=n_blocks_axis, labels=False, duplicates="drop")
    return (x_bins.astype(int) * n_blocks_axis + y_bins.astype(int)).astype(int)

def tune_xgb_optuna(table: pd.DataFrame, feature_cols: list[str], cub_class: str, n_trials: int = 100) -> dict | None:
    """Tune XGBoost with Optuna using coarse spatial-block validation."""
    if not (HAS_XGBOOST and HAS_OPTUNA):
        logger.warning("%s XGBoost tuning skipped because xgboost/optuna is missing.", cub_class)
        return None
    blocks = spatial_block_ids(table)
    unique_blocks = sorted(blocks.unique())
    X = table[feature_cols]
    y = table["label"].astype(int)

    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=50),
            "max_depth": trial.suggest_int("max_depth", 2, 8),
            "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10, log=True),
            "tree_method": "hist",
            "eval_metric": "auc",
            "random_state": RANDOM_SEED,
            "n_jobs": -1,
        }
        aucs = []
        for block in unique_blocks:
            train_idx = blocks != block
            test_idx = blocks == block
            if y[test_idx].nunique() < 2 or y[train_idx].nunique() < 2:
                continue
            model = fit_model("xgb", X.loc[train_idx], y.loc[train_idx], params)
            score = predict_positive_proba(model, X.loc[test_idx])
            aucs.append(roc_auc_score(y.loc[test_idx], score))
        return float(np.mean(aucs)) if aucs else 0.5

    study = optuna.create_study(direction="maximize", study_name=f"xgb_{cub_class}")
    study.optimize(objective, n_trials=n_trials)
    best = dict(study.best_params)
    best.update({"tree_method": "hist", "eval_metric": "auc", "random_state": RANDOM_SEED, "n_jobs": -1})
    logger.info("%s XGBoost best Optuna AUC=%.3f params=%s", cub_class, study.best_value, best)
    return best

tuned_params = {}
for cub_class, data in model_data.items():
    table = data["table"]
    feature_cols = data["feature_cols"]
    if RUN_TUNING:
        rf_params = tune_rf_random_search(table, feature_cols, cub_class, n_iter=50)
        xgb_params = tune_xgb_optuna(table, feature_cols, cub_class, n_trials=N_OPTUNA_TRIALS)
    else:
        rf_params = dict(DEFAULT_RF_PARAMS)
        xgb_params = dict(DEFAULT_XGB_PARAMS) if HAS_XGBOOST else None
    tuned_params[cub_class] = {"rf": rf_params, "xgb": xgb_params}

with (MODEL_DIR / "tuned_params.json").open("w", encoding="utf-8") as f:
    json.dump(tuned_params, f, indent=2, default=str)
display(tuned_params)


## 10. Final model training and LOIO evaluation

This section trains fold models for honest LOIO metrics and production models on
all available GPS presences plus background. The production models are the ones
used for map prediction.


In [ ]:
all_cv_results = []
final_models = {}
fold_models = {}

for cub_class, data in model_data.items():
    table = data["table"]
    feature_cols = data["feature_cols"]
    final_models[cub_class] = {}
    fold_models[cub_class] = {}

    rf_params = tuned_params[cub_class]["rf"]
    rf_results, rf_fold_models = evaluate_loio("rf", rf_params, table, feature_cols, cub_class, random_state=RANDOM_SEED)
    all_cv_results.append(rf_results)
    fold_models[cub_class]["rf"] = rf_fold_models
    final_rf = fit_model("rf", table[feature_cols], table["label"].astype(int), rf_params)
    final_models[cub_class]["rf"] = final_rf

    rf_model_path = MODEL_DIR / f"rf_model_{cub_class}_final.pkl"
    if HAS_JOBLIB:
        joblib.dump(final_rf, rf_model_path)
    else:
        with rf_model_path.open("wb") as f:
            pickle.dump(final_rf, f)
    logger.info("Saved RF production model: %s", rf_model_path)

    xgb_params = tuned_params[cub_class].get("xgb")
    if HAS_XGBOOST and xgb_params:
        xgb_results, xgb_fold_models = evaluate_loio("xgb", xgb_params, table, feature_cols, cub_class, random_state=RANDOM_SEED)
        all_cv_results.append(xgb_results)
        fold_models[cub_class]["xgb"] = xgb_fold_models
        final_xgb = fit_model("xgb", table[feature_cols], table["label"].astype(int), xgb_params)
        final_models[cub_class]["xgb"] = final_xgb
        xgb_model_path = MODEL_DIR / f"xgb_model_{cub_class}_final.json"
        final_xgb.save_model(xgb_model_path)
        logger.info("Saved XGBoost production model: %s", xgb_model_path)
    else:
        logger.warning("%s: XGBoost skipped; install xgboost to run the primary model.", cub_class)

cv_results = pd.concat(all_cv_results, ignore_index=True)
cv_results.to_csv(TABLE_DIR / "loio_cv_results.csv", index=False)
display(cv_results)


## 11. Feature importance

Random Forest importance is always available through mean decrease in impurity
and permutation importance. SHAP-based XGBoost importance runs when `shap` and
`xgboost` are installed and `RUN_SHAP=True`.


In [ ]:
importance_tables = []

def compute_rf_importance(cub_class: str, model, table: pd.DataFrame, feature_cols: list[str]) -> pd.DataFrame:
    """Compute RF MDI and permutation importance."""
    X = table[feature_cols]
    y = table["label"].astype(int)
    perm = permutation_importance(model, X, y, n_repeats=10, random_state=RANDOM_SEED, n_jobs=-1, scoring="roc_auc")
    out = pd.DataFrame(
        {
            "cub_class": cub_class,
            "algorithm": "rf",
            "feature": feature_cols,
            "mdi_importance": model.feature_importances_,
            "permutation_importance_mean": perm.importances_mean,
            "permutation_importance_std": perm.importances_std,
        }
    ).sort_values("permutation_importance_mean", ascending=False)
    return out

def compute_xgb_shap_importance(cub_class: str, model, table: pd.DataFrame, feature_cols: list[str]) -> pd.DataFrame | None:
    """Compute SHAP importance for the XGBoost production model."""
    if not (RUN_SHAP and HAS_SHAP and HAS_XGBOOST):
        return None
    X = table[feature_cols]
    sample = X.sample(min(5000, len(X)), random_state=RANDOM_SEED)
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(sample)
    mean_abs = np.abs(shap_values).mean(axis=0)

    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values, sample, show=False)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / f"shap_beeswarm_{cub_class}.png", dpi=300)
    plt.close()

    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values, sample, plot_type="bar", show=False)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / f"shap_bar_{cub_class}.png", dpi=300)
    plt.close()

    top_features = [feature_cols[i] for i in np.argsort(mean_abs)[::-1][:5]]
    for feature in top_features:
        shap.dependence_plot(feature, shap_values, sample, interaction_index="auto", show=False)
        plt.tight_layout()
        plt.savefig(PLOT_DIR / f"shap_dependence_{cub_class}_{feature}.png", dpi=300)
        plt.close()

    return pd.DataFrame(
        {
            "cub_class": cub_class,
            "algorithm": "xgb",
            "feature": feature_cols,
            "mean_abs_shap": mean_abs,
        }
    ).sort_values("mean_abs_shap", ascending=False)

for cub_class, data in model_data.items():
    table = data["table"]
    feature_cols = data["feature_cols"]
    rf_imp = compute_rf_importance(cub_class, final_models[cub_class]["rf"], table, feature_cols)
    importance_tables.append(rf_imp)
    if "xgb" in final_models[cub_class]:
        shap_imp = compute_xgb_shap_importance(cub_class, final_models[cub_class]["xgb"], table, feature_cols)
        if shap_imp is not None:
            importance_tables.append(shap_imp)

feature_importance = pd.concat(importance_tables, ignore_index=True)
feature_importance.to_csv(TABLE_DIR / "feature_importance.csv", index=False)
feature_importance.to_csv(TABLE_DIR / "feature_importance_shap.csv", index=False)
display(feature_importance.head(30))


## 12. Full-study-area prediction and zone classification

Prediction is batch-based to avoid loading the full raster stack as one huge
array. Set `RUN_MAP_PREDICTION=True` for the full map outputs. Zone thresholds
follow the CARTOBIO method: adequate, good, and optimal thresholds are derived
from suitability values at training presences.


In [ ]:
def raster_lookup_from_inventory(inventory: pd.DataFrame) -> dict[str, Path]:
    """Map feature names to raster paths."""
    return {row.feature: Path(row.path) for row in inventory.itertuples(index=False)}

def predict_raster_stack(
    model,
    inventory: pd.DataFrame,
    feature_cols: list[str],
    medians: pd.Series,
    log_columns: list[str],
    output_path: Path,
    batch_size: int = 100_000,
) -> Path:
    """Predict habitat suitability for every valid pixel in the raster stack."""
    lookup = raster_lookup_from_inventory(inventory)
    paths = [lookup[col] for col in feature_cols]
    with rasterio.open(paths[0]) as template:
        height, width = template.height, template.width
        profile = template.profile.copy()
        profile.update(count=1, dtype="float32", nodata=-9999.0, compress="lzw")
        rows_per_batch = max(1, int(batch_size // width))

    output_path.parent.mkdir(parents=True, exist_ok=True)
    sources = [rasterio.open(path) for path in paths]
    try:
        with rasterio.open(output_path, "w", **profile) as dst:
            for row_start in range(0, height, rows_per_batch):
                n_rows = min(rows_per_batch, height - row_start)
                window = Window(0, row_start, width, n_rows)
                bands = []
                for feature, src in zip(feature_cols, sources):
                    arr = src.read(1, window=window).astype("float64")
                    if src.nodata is not None:
                        arr[np.isclose(arr, src.nodata)] = np.nan
                    arr[~np.isfinite(arr)] = np.nan
                    if feature in log_columns:
                        arr[arr < 0] = np.nan
                        arr = np.log10(arr + 1.0)
                    bands.append(arr.reshape(-1))
                X = pd.DataFrame(np.column_stack(bands), columns=feature_cols)
                nan_fraction = X.isna().mean(axis=1)
                valid = nan_fraction <= 0.20
                preds = np.full(len(X), -9999.0, dtype="float32")
                if valid.any():
                    X_valid = X.loc[valid].fillna(medians)
                    preds[valid.to_numpy()] = predict_positive_proba(model, X_valid).astype("float32")
                dst.write(preds.reshape(n_rows, width), 1, window=window)
    finally:
        for src in sources:
            src.close()
    logger.info("Wrote suitability raster: %s", output_path)
    return output_path

def sample_raster_at_points(raster_path: Path, points: pd.DataFrame) -> np.ndarray:
    """Sample one raster at projected point coordinates."""
    coords = list(zip(points["x_3035"].astype(float), points["y_3035"].astype(float)))
    with rasterio.open(raster_path) as src:
        values = np.array([sample[0] for sample in src.sample(coords)], dtype=float)
        if src.nodata is not None:
            values[np.isclose(values, src.nodata)] = np.nan
        values[~np.isfinite(values)] = np.nan
    return values

def compute_zone_thresholds(presence_suitability: np.ndarray) -> dict:
    """Compute CARTOBIO adequate, good, and optimal thresholds from presences."""
    scores = np.asarray(presence_suitability, dtype=float)
    scores = scores[np.isfinite(scores)]
    if scores.size == 0:
        raise ValueError("No finite presence suitability values for thresholding.")
    p10 = np.percentile(scores, 10)
    adequate = float(scores[scores <= p10].mean())
    good = float(scores[scores >= adequate].mean())
    optimal = float(scores[scores >= good].mean())
    return {"adequate": adequate, "good": good, "optimal": optimal}

def remove_small_patches(binary_raster: np.ndarray, min_area_km2: float = 0.5, pixel_size_m: float = 100) -> np.ndarray:
    """Remove connected patches smaller than the minimum ecological area."""
    min_pixels = max(1, int(math.ceil((min_area_km2 * 1_000_000) / (pixel_size_m**2))))
    labeled, n_labels = ndimage.label(binary_raster.astype(bool), structure=np.ones((3, 3), dtype=bool))
    if n_labels == 0:
        return binary_raster.astype(bool)
    counts = np.bincount(labeled.ravel())
    keep = np.zeros_like(counts, dtype=bool)
    keep[counts >= min_pixels] = True
    keep[0] = False
    return keep[labeled]

def classify_zone_raster(
    suitability_path: Path,
    presence_points: pd.DataFrame,
    output_path: Path,
    min_area_km2: float = 0.5,
) -> tuple[Path, dict, pd.DataFrame]:
    """Classify continuous suitability into none, adequate, good, and optimal zones."""
    presence_scores = sample_raster_at_points(suitability_path, presence_points)
    thresholds = compute_zone_thresholds(presence_scores)
    with rasterio.open(suitability_path) as src:
        suitability = src.read(1).astype(float)
        nodata = src.nodata
        valid = np.isfinite(suitability)
        if nodata is not None:
            valid &= ~np.isclose(suitability, nodata)
        profile = src.profile.copy()
        profile.update(dtype="uint8", nodata=255, compress="lzw")
        pixel_area_km2 = abs(src.transform.a * src.transform.e) / 1_000_000

    adequate_mask = valid & (suitability >= thresholds["adequate"])
    good_mask = valid & (suitability >= thresholds["good"])
    optimal_mask = valid & (suitability >= thresholds["optimal"])
    adequate_mask = remove_small_patches(adequate_mask, min_area_km2=min_area_km2)
    good_mask = remove_small_patches(good_mask, min_area_km2=min_area_km2)
    optimal_mask = remove_small_patches(optimal_mask, min_area_km2=min_area_km2)

    zones = np.full(suitability.shape, 255, dtype="uint8")
    zones[valid] = 0
    zones[adequate_mask] = 1
    zones[good_mask] = 2
    zones[optimal_mask] = 3

    with rasterio.open(output_path, "w", **profile) as dst:
        dst.write(zones, 1)

    area_records = []
    labels = {1: "adequate", 2: "good", 3: "optimal"}
    for value, label in labels.items():
        area_records.append({"zone_value": value, "zone": label, "area_km2": float((zones == value).sum() * pixel_area_km2)})
    areas = pd.DataFrame(area_records)
    logger.info("Wrote zones raster: %s thresholds=%s", output_path, thresholds)
    return output_path, thresholds, areas

def plot_raster_map(path: Path, title: str, output_path: Path, cmap: str = "viridis") -> None:
    """Save a simple map figure for a raster output."""
    with rasterio.open(path) as src:
        arr = src.read(1, masked=True)
        extent = (src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top)
    fig, ax = plt.subplots(figsize=(10, 6))
    image = ax.imshow(arr, extent=extent, origin="upper", cmap=cmap)
    ax.set_title(title)
    ax.set_xlabel(f"X ({TARGET_CRS})")
    ax.set_ylabel(f"Y ({TARGET_CRS})")
    fig.colorbar(image, ax=ax, fraction=0.035, pad=0.02)
    add_scalebar(ax)
    fig.tight_layout()
    fig.savefig(output_path, dpi=300)
    plt.close(fig)

REFERENCE_ZONE_AREAS = pd.DataFrame(
    [
        {"cub_class": "lt6", "zone": "adequate", "paper_threshold": 0.146, "paper_area_km2": 5927.8},
        {"cub_class": "lt6", "zone": "good", "paper_threshold": 0.388, "paper_area_km2": 2363.1},
        {"cub_class": "lt6", "zone": "optimal", "paper_threshold": 0.619, "paper_area_km2": 1789.0},
        {"cub_class": "6to12", "zone": "adequate", "paper_threshold": 0.139, "paper_area_km2": 6805.2},
        {"cub_class": "6to12", "zone": "good", "paper_threshold": 0.400, "paper_area_km2": 2613.7},
        {"cub_class": "6to12", "zone": "optimal", "paper_threshold": 0.649, "paper_area_km2": 2147.8},
    ]
)


In [ ]:
map_outputs = []
zone_area_tables = []
zone_threshold_tables = []

if RUN_MAP_PREDICTION:
    for cub_class, data in model_data.items():
        for algo, model in final_models[cub_class].items():
            feature_cols = data["feature_cols"]
            suitability_path = MAP_DIR / f"suitability_{cub_class}_{algo}.tif"
            predict_raster_stack(
                model,
                raster_inventory,
                feature_cols,
                data["medians"],
                data["log_columns"],
                suitability_path,
                batch_size=BATCH_SIZE,
            )
            zones_path, thresholds, areas = classify_zone_raster(
                suitability_path,
                presence_sets[cub_class],
                MAP_DIR / f"zones_{cub_class}_{algo}.tif",
            )
            plot_raster_map(suitability_path, f"Suitability {cub_class} {algo}", PLOT_DIR / f"suitability_{cub_class}_{algo}.png")
            plot_raster_map(zones_path, f"Habitat zones {cub_class} {algo}", PLOT_DIR / f"zones_{cub_class}_{algo}.png", cmap="plasma")
            map_outputs.append({"cub_class": cub_class, "algorithm": algo, "suitability_path": suitability_path, "zones_path": zones_path})
            areas.insert(0, "algorithm", algo)
            areas.insert(0, "cub_class", cub_class)
            zone_area_tables.append(areas)
            zone_threshold_tables.append({"cub_class": cub_class, "algorithm": algo, **thresholds})

            if RUN_UNCERTAINTY_MAPS and algo in fold_models[cub_class]:
                fold_paths = []
                for fold_record in fold_models[cub_class][algo]:
                    fold_path = MAP_DIR / f"suitability_{cub_class}_{algo}_fold_{fold_record['fold']}.tif"
                    predict_raster_stack(
                        fold_record["model"],
                        raster_inventory,
                        feature_cols,
                        data["medians"],
                        data["log_columns"],
                        fold_path,
                        batch_size=BATCH_SIZE,
                    )
                    fold_paths.append(fold_path)
                # Pixel-wise std is computed in a streaming row loop.
                std_path = MAP_DIR / f"suitability_{cub_class}_{algo}_std.tif"
                with rasterio.open(fold_paths[0]) as template:
                    profile = template.profile.copy()
                    profile.update(dtype="float32", nodata=-9999.0, compress="lzw")
                    height, width = template.height, template.width
                    rows_per_batch = max(1, int(BATCH_SIZE // width))
                sources = [rasterio.open(path) for path in fold_paths]
                try:
                    with rasterio.open(std_path, "w", **profile) as dst:
                        for row_start in range(0, height, rows_per_batch):
                            n_rows = min(rows_per_batch, height - row_start)
                            window = Window(0, row_start, width, n_rows)
                            stack = np.stack([src.read(1, window=window).astype(float) for src in sources], axis=0)
                            stack[np.isclose(stack, -9999.0)] = np.nan
                            std = np.nanstd(stack, axis=0).astype("float32")
                            std[~np.isfinite(std)] = -9999.0
                            dst.write(std, 1, window=window)
                finally:
                    for src in sources:
                        src.close()
                plot_raster_map(std_path, f"Prediction uncertainty {cub_class} {algo}", PLOT_DIR / f"suitability_{cub_class}_{algo}_std.png", cmap="magma")
else:
    logger.info("RUN_MAP_PREDICTION=False; suitability and zone rasters were not generated in this run.")

if zone_area_tables:
    zone_areas = pd.concat(zone_area_tables, ignore_index=True)
    zone_areas = zone_areas.merge(REFERENCE_ZONE_AREAS, on=["cub_class", "zone"], how="left")
    zone_areas["diff_percent_vs_paper"] = 100 * (zone_areas["area_km2"] - zone_areas["paper_area_km2"]) / zone_areas["paper_area_km2"]
    zone_areas.to_csv(TABLE_DIR / "zone_areas.csv", index=False)
    zone_thresholds = pd.DataFrame(zone_threshold_tables)
    zone_thresholds.to_csv(TABLE_DIR / "zone_thresholds.csv", index=False)
    display(zone_areas)
    display(zone_thresholds)


## 13. Comparison with original MaxEnt rasters

The supplied MaxEnt outputs are already aligned in shape, transform, and bounds
with the project grid, though rasterio may report a non-identical CRS authority
string. These comparisons use continuous map overlap metrics plus zone agreement.


In [ ]:
def read_raster_as_template(path: Path, template_path: Path, categorical: bool = False) -> np.ma.MaskedArray:
    """Read a raster, reprojecting to a template only when geometry differs."""
    with rasterio.open(template_path) as template, rasterio.open(path) as src:
        same_geometry = (
            src.width == template.width
            and src.height == template.height
            and src.transform.almost_equals(template.transform)
            and np.allclose(tuple(src.bounds), tuple(template.bounds), atol=1e-6)
        )
        if same_geometry:
            arr = src.read(1, masked=True)
        else:
            destination = np.full((template.height, template.width), template.nodata or -9999.0, dtype="float32")
            reproject(
                source=rasterio.band(src, 1),
                destination=destination,
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=template.transform,
                dst_crs=template.crs,
                resampling=Resampling.nearest if categorical else Resampling.bilinear,
                dst_nodata=template.nodata or -9999.0,
            )
            arr = np.ma.masked_invalid(destination)
    return arr

def schoeners_D(map1: np.ndarray, map2: np.ndarray) -> float:
    """Compute Schoener's D overlap between two suitability maps."""
    a = np.asarray(map1, dtype=float)
    b = np.asarray(map2, dtype=float)
    valid = np.isfinite(a) & np.isfinite(b) & (a >= 0) & (b >= 0)
    if valid.sum() == 0:
        return np.nan
    p1 = a[valid] / np.nansum(a[valid])
    p2 = b[valid] / np.nansum(b[valid])
    return float(1 - 0.5 * np.sum(np.abs(p1 - p2)))

def warrens_I(map1: np.ndarray, map2: np.ndarray) -> float:
    """Compute Warren's I overlap between two suitability maps."""
    a = np.asarray(map1, dtype=float)
    b = np.asarray(map2, dtype=float)
    valid = np.isfinite(a) & np.isfinite(b) & (a >= 0) & (b >= 0)
    if valid.sum() == 0:
        return np.nan
    p1 = a[valid] / np.nansum(a[valid])
    p2 = b[valid] / np.nansum(b[valid])
    return float(1 - 0.5 * np.sqrt(np.sum((np.sqrt(p1) - np.sqrt(p2)) ** 2)))

def compare_with_maxent(cub_class: str, algo: str, suitability_path: Path, zones_path: Path) -> dict:
    """Compare our model rasters against the original MaxEnt products."""
    refs = REFERENCE_MAXENT[cub_class]
    if not refs["model"].exists() or not refs["zones"].exists():
        logger.warning("Missing MaxEnt reference for %s", cub_class)
        return {}
    ours = read_raster_as_template(suitability_path, suitability_path).astype(float)
    maxent = read_raster_as_template(refs["model"], suitability_path).astype(float)
    ours_zones = read_raster_as_template(zones_path, zones_path, categorical=True).astype(float)
    maxent_zones = read_raster_as_template(refs["zones"], zones_path, categorical=True).astype(float)

    ours_data = ours.filled(np.nan)
    maxent_data = maxent.filled(np.nan)
    valid = np.isfinite(ours_data) & np.isfinite(maxent_data) & (ours_data >= 0) & (maxent_data >= 0)
    pearson = pearsonr(maxent_data[valid], ours_data[valid]).statistic if valid.sum() > 2 else np.nan
    spearman = spearmanr(maxent_data[valid], ours_data[valid]).correlation if valid.sum() > 2 else np.nan

    oz = ours_zones.filled(np.nan)
    mz = maxent_zones.filled(np.nan)
    zvalid = np.isfinite(oz) & np.isfinite(mz) & (oz != 255) & (mz != 255)
    kappa = cohen_kappa_score(mz[zvalid].astype(int), oz[zvalid].astype(int)) if zvalid.sum() else np.nan

    sample_idx = np.flatnonzero(valid)
    if sample_idx.size:
        rng = np.random.default_rng(RANDOM_SEED)
        sample_idx = rng.choice(sample_idx, size=min(100_000, sample_idx.size), replace=False)
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.scatter(maxent_data.ravel()[sample_idx], ours_data.ravel()[sample_idx], s=1, alpha=0.15)
        ax.set_xlabel("MaxEnt suitability")
        ax.set_ylabel(f"{algo} suitability")
        ax.set_title(f"MaxEnt vs {algo}: {cub_class}")
        fig.tight_layout()
        fig.savefig(PLOT_DIR / f"maxent_scatter_{cub_class}_{algo}.png", dpi=300)
        plt.close(fig)

    return {
        "cub_class": cub_class,
        "algorithm": algo,
        "pearson_r": pearson,
        "spearman_rho": spearman,
        "cohen_kappa_zones": kappa,
        "schoeners_D": schoeners_D(ours_data, maxent_data),
        "warrens_I": warrens_I(ours_data, maxent_data),
    }

comparison_records = []
if RUN_MAP_PREDICTION:
    for record in map_outputs:
        comparison = compare_with_maxent(record["cub_class"], record["algorithm"], record["suitability_path"], record["zones_path"])
        if comparison:
            comparison_records.append(comparison)

if comparison_records:
    model_comparison = pd.DataFrame(comparison_records)
    model_comparison.to_csv(TABLE_DIR / "model_comparison.csv", index=False)
    display(model_comparison)
else:
    logger.info("MaxEnt comparison skipped because map prediction outputs are unavailable in this run.")


## 14. Independent OBS validation

This section evaluates GPS-trained production models on observation records that
were never used for training. The OBS records are compared against an equal-sized
background sample to test whether the model ranks observed locations as more
suitable than available locations.


In [ ]:
def build_external_validation_table(
    obs_points: pd.DataFrame,
    background: pd.DataFrame,
    inventory: pd.DataFrame,
    feature_cols: list[str],
    log_columns: list[str],
    medians: pd.Series,
    n_background: int | None = None,
) -> pd.DataFrame:
    """Extract features for OBS presences plus random background validation points."""
    n_background = n_background or len(obs_points)
    bg = background.sample(min(n_background, len(background)), random_state=RANDOM_SEED).copy()
    obs = obs_points.copy()
    obs["label"] = 1
    obs["source"] = "obs"
    obs["point_id"] = obs["id_obs"].astype("string")
    bg["label"] = 0
    points = pd.concat(
        [
            obs[["point_id", "x_3035", "y_3035", "source", "label"]],
            bg[["point_id", "x_3035", "y_3035", "source", "label"]],
        ],
        ignore_index=True,
    )
    extracted = extract_raster_values(points, inventory)
    extracted = extracted[feature_cols]
    transformed = apply_feature_transforms(extracted, log_columns)
    keep = transformed.isna().mean(axis=1) <= 0.20
    table = pd.concat(
        [
            points.loc[keep].reset_index(drop=True),
            transformed.loc[keep].reset_index(drop=True).fillna(medians),
        ],
        axis=1,
    )
    return table

obs_validation_records = []
for cub_class, data in model_data.items():
    obs_subset = obs_clean[obs_clean["cub_class"] == cub_class].copy()
    if obs_subset.empty:
        logger.warning("No OBS validation records for %s", cub_class)
        continue
    # Thin OBS only for validation density control, not for training.
    obs_subset = spatial_thin_one(obs_subset, min_dist_m=MIN_THIN_DISTANCE_M, n_iterations=N_THIN_ITERATIONS, random_state=RANDOM_SEED)
    validation_table = build_external_validation_table(
        obs_subset,
        background_points,
        raster_inventory,
        data["feature_cols"],
        data["log_columns"],
        data["medians"],
        n_background=len(obs_subset),
    )
    for algo, model in final_models[cub_class].items():
        scores = predict_positive_proba(model, validation_table[data["feature_cols"]])
        y = validation_table["label"].astype(int).to_numpy()
        metrics = evaluate_predictions(y, scores)
        obs_scores = scores[y == 1]
        bg_scores = scores[y == 0]
        try:
            mw = mannwhitneyu(obs_scores, bg_scores, alternative="greater")
            mw_p = float(mw.pvalue)
        except Exception:
            mw_p = np.nan
        obs_validation_records.append(
            {
                "cub_class": cub_class,
                "algorithm": algo,
                "n_obs_presence": int((y == 1).sum()),
                "n_background": int((y == 0).sum()),
                "mean_obs_suitability": float(np.nanmean(obs_scores)),
                "mean_background_suitability": float(np.nanmean(bg_scores)),
                "mannwhitney_p_greater": mw_p,
                **metrics,
            }
        )

obs_validation = pd.DataFrame(obs_validation_records)
obs_validation.to_csv(TABLE_DIR / "obs_validation.csv", index=False)
display(obs_validation)


## 15. Optional OBS-trained parallel models

These models are not the primary deliverable. They are useful for checking how
much the map changes if the biased OBS data are treated as presences. Leave this
off for the first runs unless you specifically want the diagnostic maps.


In [ ]:
obs_parallel_records = []

if RUN_OBS_PARALLEL_MODELS:
    for cub_class, data in model_data.items():
        obs_subset = obs_clean[obs_clean["cub_class"] == cub_class].copy()
        if obs_subset.empty:
            continue
        obs_subset = spatial_thin_one(obs_subset, min_dist_m=MIN_THIN_DISTANCE_M, n_iterations=N_THIN_ITERATIONS, random_state=RANDOM_SEED)
        obs_subset["bear_name"] = obs_subset["confirmed_individual"].astype("string").fillna("obs_unknown")
        obs_subset["id_obs"] = obs_subset["id_obs"].astype("string")
        obs_model_data = prepare_training_table(obs_subset, background_points, raster_inventory, cub_class=f"{cub_class}_obs")
        rf_params = tuned_params[cub_class]["rf"]
        obs_rf = fit_model("rf", obs_model_data["table"][obs_model_data["feature_cols"]], obs_model_data["table"]["label"].astype(int), rf_params)
        obs_parallel_records.append({"cub_class": cub_class, "algorithm": "rf", "n_obs_presences": int((obs_model_data["table"]["label"] == 1).sum())})
        if RUN_MAP_PREDICTION:
            obs_path = MAP_DIR / f"suitability_{cub_class}_obs_rf.tif"
            predict_raster_stack(
                obs_rf,
                raster_inventory,
                obs_model_data["feature_cols"],
                obs_model_data["medians"],
                obs_model_data["log_columns"],
                obs_path,
                batch_size=BATCH_SIZE,
            )
else:
    logger.info("RUN_OBS_PARALLEL_MODELS=False; OBS-trained diagnostic models were skipped.")

obs_parallel_summary = pd.DataFrame(obs_parallel_records)
if not obs_parallel_summary.empty:
    obs_parallel_summary.to_csv(TABLE_DIR / "obs_parallel_models.csv", index=False)
    display(obs_parallel_summary)


## 16. Output manifest

The final cell writes a manifest of generated outputs and shows the main tables.
If map prediction was disabled, this still records all intermediate tables,
models, plots, and logs produced by the lighter run.


In [ ]:
def output_manifest(output_root: Path) -> pd.DataFrame:
    """List generated files under the output root."""
    records = []
    for path in sorted(output_root.rglob("*")):
        if path.is_file():
            records.append(
                {
                    "path": str(path.relative_to(output_root)),
                    "size_mb": path.stat().st_size / 1_000_000,
                    "modified_utc": pd.to_datetime(path.stat().st_mtime, unit="s", utc=True),
                }
            )
    return pd.DataFrame(records)

manifest = output_manifest(OUTPUT_ROOT)
manifest.to_csv(OUTPUT_ROOT / "output_manifest.csv", index=False)
logger.info("Notebook workflow complete. Generated %d output files.", len(manifest))
display(manifest)
